# Fine-tune German DistilBERT on 10kGNAD (Colab)

Part of [german-text-classifier](https://github.com/abhisheklakhani-it/german-text-classifier).

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. Training takes about 15 minutes. The last cell downloads `distilbert_results.zip`.
3. Unzip it into the project folder: it contains `models/distilbert/` and `results/transformer*.json(l)`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install transformers==5.19.0 datasets==5.1.0 accelerate==1.15.0 scikit-learn==1.9.1

In [ ]:
!mkdir -p src/gnad scripts results models

In [ ]:
%%writefile src/gnad/__init__.py

In [ ]:
%%writefile src/gnad/data.py
"""Load 10kGNAD and split it into train / validation / test.

The test split is the official one (1,028 articles) and is only used for the final
comparison. Model choices (grid search, epochs) use a stratified 10% validation split
carved out of the official training data.
"""

from dataclasses import dataclass

from datasets import load_dataset
from sklearn.model_selection import train_test_split

DATASET = "community-datasets/gnad10"
SEED = 42


@dataclass
class Split:
    texts: list[str]
    labels: list[int]

    def __len__(self) -> int:
        return len(self.texts)


@dataclass
class GNAD:
    train: Split
    validation: Split
    test: Split
    label_names: list[str]


def load(validation_size: float = 0.1) -> GNAD:
    ds = load_dataset(DATASET)
    names = ds["train"].features["label"].names
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        ds["train"]["text"], ds["train"]["label"], test_size=validation_size, stratify=ds["train"]["label"], random_state=SEED
    )
    return GNAD(
        train=Split(list(train_texts), list(train_labels)),
        validation=Split(list(val_texts), list(val_labels)),
        test=Split(list(ds["test"]["text"]), list(ds["test"]["label"])),
        label_names=names,
    )

In [ ]:
%%writefile src/gnad/metrics.py
"""Accuracy, macro-F1, per-class scores and confusion matrix, shared by both models."""

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support


def evaluate(y_true: list[int], y_pred: list[int], label_names: list[str]) -> dict:
    labels = list(range(len(label_names)))
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "per_class": {
            name: {"precision": float(precision[i]), "recall": float(recall[i]), "f1": float(f1[i]), "support": int(support[i])}
            for i, name in enumerate(label_names)
        },
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }

In [ ]:
%%writefile scripts/train_transformer.py
"""Fine-tune distilbert-base-german-cased on 10kGNAD.

    python scripts/train_transformer.py              # full run (about 15 min on a Colab T4 GPU)
    python scripts/train_transformer.py --smoke      # 20 steps on 200 articles, to test the pipeline

Picks the epoch with the best validation macro-F1, then scores it once on the test split.
"""

import argparse
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer,
                          TrainingArguments, set_seed)

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from gnad.data import SEED, load  # noqa: E402
from gnad.metrics import evaluate  # noqa: E402

BASE_MODEL = "distilbert-base-german-cased"


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--smoke", action="store_true")
    parser.add_argument("--epochs", type=float, default=3)
    parser.add_argument("--max-length", type=int, default=512)
    parser.add_argument("--lr", type=float, default=5e-5)
    args = parser.parse_args()
    set_seed(SEED)

    data = load()
    names = data.label_names
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

    def to_dataset(split, limit=None):
        texts, labels = split.texts[:limit], split.labels[:limit]
        ds = Dataset.from_dict({"text": texts, "label": labels})
        return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=args.max_length), batched=True, remove_columns=["text"])

    limit = 200 if args.smoke else None
    train, val, test = to_dataset(data.train, limit), to_dataset(data.validation, limit), to_dataset(data.test, limit)
    truncated = sum(len(ids) >= args.max_length for ids in to_dataset(data.test)["input_ids"]) if not args.smoke else None

    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=len(names), id2label=dict(enumerate(names)), label2id={n: i for i, n in enumerate(names)}
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        result = evaluate(labels.tolist(), np.argmax(logits, axis=-1).tolist(), names)
        return {"accuracy": result["accuracy"], "macro_f1": result["macro_f1"]}

    out_dir = ROOT / ("checkpoints/smoke" if args.smoke else "checkpoints/distilbert")
    training_args = TrainingArguments(
        output_dir=str(out_dir),
        num_train_epochs=args.epochs,
        max_steps=20 if args.smoke else -1,
        learning_rate=args.lr,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,  # effective batch 16, fits in 8 GB unified memory
        per_device_eval_batch_size=16,
        warmup_steps=0.1,  # a float below 1 is a ratio of the total steps
        weight_decay=0.01,
        eval_strategy="steps" if args.smoke else "epoch",
        eval_steps=10 if args.smoke else None,
        save_strategy="steps" if args.smoke else "epoch",
        save_steps=10 if args.smoke else None,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        train_sampling_strategy="group_by_length",  # batches of similar length waste less padding
        logging_steps=50,
        report_to=[],
        fp16=torch.cuda.is_available(),  # half precision on an NVIDIA GPU (Colab); MPS and CPU stay fp32
        seed=SEED,
    )
    trainer = Trainer(model=model, args=training_args, train_dataset=train, eval_dataset=val,
                      data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
    start = time.perf_counter()
    trainer.train()
    train_minutes = (time.perf_counter() - start) / 60

    start = time.perf_counter()
    predictions = trainer.predict(test)
    test_seconds = time.perf_counter() - start
    pred = np.argmax(predictions.predictions, axis=-1).tolist()
    probs = torch.softmax(torch.tensor(predictions.predictions), dim=-1).numpy()

    result = evaluate(test["label"], pred, names)
    result.update(
        model=BASE_MODEL, device=str(trainer.args.device), max_length=args.max_length, epochs=args.epochs,
        learning_rate=args.lr, train_minutes=round(train_minutes, 1), test_seconds=round(test_seconds, 1),
        truncated_test_articles=truncated,
        validation_history=[h for h in trainer.state.log_history if "eval_macro_f1" in h],
        best_checkpoint=trainer.state.best_model_checkpoint,
    )
    tag = "transformer_smoke" if args.smoke else "transformer"
    (ROOT / "results").mkdir(exist_ok=True)
    (ROOT / "results" / f"{tag}.json").write_text(json.dumps(result, indent=2) + "\n")
    with open(ROOT / "results" / f"{tag}.test_predictions.jsonl", "w") as f:
        for i, (gold, p, pr) in enumerate(zip(test["label"], pred, probs)):
            f.write(json.dumps({"index": i, "label": gold, "pred": p, "confidence": round(float(pr.max()), 4)}) + "\n")

    if not args.smoke:
        save_dir = ROOT / "models" / "distilbert"
        trainer.save_model(str(save_dir))
        tokenizer.save_pretrained(str(save_dir))
    print(f"{tag}: accuracy {result['accuracy']:.4f}  macro-F1 {result['macro_f1']:.4f}  ({train_minutes:.1f} min on {trainer.args.device})")


if __name__ == "__main__":
    main()

Quick check that the whole pipeline runs (20 steps on 200 articles, about 1 minute):

In [ ]:
!python scripts/train_transformer.py --smoke

The real run: 3 epochs, best epoch chosen by validation macro-F1, then one score on the test split.

In [ ]:
!python scripts/train_transformer.py

In [ ]:
!cat results/transformer.json | head -20

In [ ]:
!zip -qr distilbert_results.zip models/distilbert results/transformer.json results/transformer.test_predictions.jsonl
from google.colab import files
files.download("distilbert_results.zip")